# Load Required Libraries

This section imports the core Python libraries used throughout the workflow.

The notebook relies on geospatial processing tools for:
- raster handling,
- vector boundary operations,
- file management,
- and batch processing of geospatial datasets.

In [ ]:
import os
import glob
import rasterio
from rasterio.mask import mask
import geopandas as gpd

# Define Input and Output Paths

This section defines the locations of:
- the input raster datasets,
- the Hungary boundary shapefile,
- and the output directory for clipped rasters.

Relative paths are used to improve portability across different systems and environments.

In [ ]:
raster_folder = r"./Data/EuropeanDroughtObservatory"
shapefile_path = r"./Data/hungary_shape/hun_countrybuff.shp"
output_folder = r"./Data/EuropeanDroughtObservatory/Clipped"

os.makedirs(output_folder, exist_ok=True)


Optional: This step prints basic metadata to verify:
- raster nodata values,
- data types,
- coordinate reference systems (CRS).

Useful for debugging and ensuring consistency before processing.

In [ ]:
print("Nodata:", src.nodata)
print("Dtype:", src.dtypes)
print("Raster CRS:", src.crs)
print("Shape CRS:", gdf.crs)

## Load Study Area Boundary

In this step, the Hungary boundary shapefile is loaded using GeoPandas.

This vector layer will be used to spatially clip the raster datasets to the area of interest.

In [ ]:
gdf = gpd.read_file(shapefile_path)

## Discover Input Raster Files

This step searches for all `.tif` raster files within the input directory and its subfolders.

It ensures that all relevant datasets are detected before processing.

In [ ]:
from pathlib import Path

raster_files = list(Path(raster_folder).rglob("*.tif"))
print(f"Found {len(raster_files)} raster files")


## Clip Raster Files to Study Area

This step processes all raster files and clips them to the Hungary boundary.

Workflow:
- recursively locate all `.tif` files,
- ensure matching coordinate reference systems (CRS),
- clip each raster using the study area geometry,
- and save the results to the output directory.

Each output raster retains the original filename and updated spatial extent.

*Note: In this implementation, the GeoDataFrame is reprojected in-place if CRS differs. When working with multiple rasters in different coordinate systems, it is recommended to create a temporary reprojected copy instead to avoid cumulative transformations.*

In [ ]:
raster_files = glob.glob(os.path.join(raster_folder, "**", "*.tif"), recursive=True)

print(f"Raster files found: {len(raster_files)}")

for raster_path in raster_files:
    
    with rasterio.open(raster_path) as src:
        
        if gdf.crs != src.crs:
            gdf = gdf.to_crs(src.crs)
        
        geometries = gdf.geometry.values
        
        # Clip
        out_image, out_transform = mask(src, geometries, crop=True, filled=True, nodata=src.nodata)
        
        out_meta = src.meta.copy()
        out_meta.update({
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform
        })
        
        out_path = os.path.join(output_folder, os.path.basename(raster_path))
        
        with rasterio.open(out_path, "w", **out_meta) as dest:
            dest.write(out_image)

print("Done.")

## Select Raster Files for Seasonal Filtering

This step loads all raster files from the clipped dataset for further filtering.

In [ ]:
from pathlib import Path

raster_folder = r"./Data/EuropeanDroughtObservatory/Clipped"

all_files = list(Path(raster_folder).rglob("*.tif"))

print(f"Total files: {len(all_files)}")

### Filter Summer Months

Raster files are filtered based on acquisition dates extracted from filenames.
Only May–September (05–09) are retained.

In [ ]:
summer_months = ["05", "06", "07", "08", "09"]
summer_files = []

for f in all_files:
    fname = f.name
    date_str = fname.split("_")[3]
    month = date_str[4:6]
    
    if month in summer_months:
        summer_files.append(f)

print(f"Summer month files: {len(summer_files)}")

### Save Filtered Raster Files

Selected raster files are copied into a separate folder for further analysis.

In [ ]:
import os
import shutil
from pathlib import Path

output_folder = r"./Data/EuropeanDroughtObservatory/Summer_Rasters"
os.makedirs(output_folder, exist_ok=True)

for f in summer_files:
    dest = Path(output_folder) / f.name
    shutil.copy(f, dest)